# ðŸŽ¨ Cartoonify â€” Unified Interface

Combines **Reimagine** (FLUX.1-Kontext-dev), **Scene** (Depth ControlNet), and **Portrait**
(Canny ControlNet) into a single interface. One story. One button. Three rendering styles.

**Flow:** Write your story â†’ upload a photo â†’ pick a rendering style â†’ click Cartoonify.
Gemini builds the structured prompt silently on generate â€” no separate Build Prompt step.

**Settings modes:**
- **Default** â€” hardcoded parameters per rendering style
- **Wild** â€” Gemini reads the story and suggests parameters optimised for maximum satirical impact

| Style | Pipeline | Best for |
|---|---|---|
| **Reimagine** | FLUX.1-Kontext-dev | Creative reinterpretation â€” scene can change freely |
| **Scene** | FLUX.1-dev + ControlNet Depth | Crowds, architecture â€” spatial layout preserved |
| **Portrait** | FLUX.1-dev + ControlNet Canny | Specific person must be immediately recognisable |

**VRAM note:** Only one pipeline loads at a time. Switching modes takes ~60â€“90 s (one-time per switch).

**Requires:** Google Colab Pro â†’ Runtime â†’ Change runtime type â†’ **A100 GPU**

> Accept the [FLUX.1-Kontext-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
> on HuggingFace before running â€” it is gated separately from FLUX.1-dev.

---

## 1 â€” Confirm A100 GPU

In [ ]:
!nvidia-smi

## 2 â€” Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub
!pip install --quiet google-genai
!pip install --quiet opencv-python-headless

In [ ]:
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional â€” continue to the next cell.")

## 3 â€” Imports

In [ ]:
import gc
import json
import queue
import threading
import datetime
import os
import numpy as np
import cv2
import torch
import gradio as gr
import google.genai as genai
import google.genai.types as genai_types
from PIL import Image
from diffusers import (
    FluxPipeline,
    FluxKontextPipeline,
    FluxControlNetPipeline,
    FluxControlNetModel,
)
from transformers import pipeline as hf_pipeline

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} â€” {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'cv2    : {cv2.__version__}')
print(f'gradio : {gr.__version__}')
print(f'genai  : {genai.__version__}')

## 4 â€” Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

## 5 â€” API Tokens

**Hugging Face** â€” FLUX.1-dev and FLUX.1-Kontext-dev are both gated models.
Add a Read token to Colab Secrets as `HF_TOKEN`.

**Google Gemini** â€” Add your key to Colab Secrets as `GOOGLE_API_KEY`
([aistudio.google.com/apikey](https://aistudio.google.com/apikey)).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN       = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

login(token=HF_TOKEN, add_to_git_credential=False)
print('âœ“ Logged in to Hugging Face')
print('âœ“ Google API key loaded' if GOOGLE_API_KEY else 'âš ï¸  GOOGLE_API_KEY not found â€” story-to-prompt and Wild mode will not work')

## 6 â€” Configuration

Edit this cell to change the LoRA, trigger word, default rendering mode, or default settings mode.
Nothing else needs to change.

In [ ]:
# â”€â”€ LoRA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LORA_SOURCE     = 'drive'   # 'huggingface' | 'drive'
LORA_HF_REPO    = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT  = 'Ghibili-Cartoon-Art.safetensors'
LORA_DRIVE_PATH         = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'
LORA_GADO_ADAPTER_NAME  = 'gado_cartoon'   # adapter name passed to load_lora_weights / set_adapters

# â”€â”€ Satirefic LoRA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LORA_SATIREFIC_PATH         = '/content/drive/MyDrive/satirefic/satirefic/satirefic.safetensors'
LORA_SATIREFIC_ADAPTER_NAME = 'satirefic'

# â”€â”€ Style / LoRA selector options â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Maps UI label -> adapter name and trigger word used in prompt building.
LORA_STYLES = {
    'None':           {'adapter': None,           'trigger': ''},
    'Gado Cartoon':   {'adapter': 'gado_cartoon',  'trigger': 'gdo_cartoon'},
    'Satirefic LoRA': {'adapter': 'satirefic',     'trigger': 'satirefic'},
}

# â”€â”€ Trigger + fallback prompt â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEFAULT_TRIGGER = 'gdo_cartoon'
DEFAULT_PROMPT  = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# Global prompt prefix applied to every generation (all modes, all LoRAs, manual or Gemini)
GLOBAL_PROMPT_PREFIX  = 'crude black and white newspaper caricature'

# Satirefic hidden style boost applied only when Satirefic LoRA is selected
SATIREFIC_STYLE_BOOST = (
    'maximum grotesque political satire, extreme caricature exaggeration, '
    'absurd anti-authoritarian newspaper cartoon, rough dirty ink linework, '
    'ugly distorted faces, pig-politician symbolism, visual irony, '
    'anarchist press style, flat white paper, brutal black ink, no readable text'
)

# â”€â”€ Gemini â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
GOOGLE_MODEL = 'gemini-2.5-flash-lite'

# â”€â”€ Base models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BASE_MODEL_KONTEXT = 'black-forest-labs/FLUX.1-Kontext-dev'
BASE_MODEL_FLUX    = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL   = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'
DEPTH_MODEL        = 'depth-anything/Depth-Anything-V2-Small-hf'

# â”€â”€ Default parameters per rendering mode â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEFAULTS = {
    'imagine':   {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'reimagine': {'guidance': 2.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'scene':     {'guidance': 3.5, 'cn_scale': 0.8, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'portrait':  {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
}
DEFAULT_STEPS = 28
DEFAULT_SEED  = 42
DEFAULT_MODE  = 'Reimagine'   # Imagine | Reimagine | Scene | Portrait

# â”€â”€ Output directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Default mode : {DEFAULT_MODE}')
print(f'LoRA source  : {LORA_SOURCE}')
print(f'Output dir   : {OUTPUT_DIR}')


## 7 â€” Load Models

`load_pipeline(mode)` handles VRAM switching â€” unloads the current pipeline before loading the new one.
The initial load uses `DEFAULT_MODE` set in cell-config.

â±ï¸ First run downloads ~24â€“29 GB depending on mode. Warm cache: ~1 minute.

In [ ]:
active_mode            = None
pipe                   = None
controlnet             = None
depth_estimator        = None
_satirefic_lora_loaded = False


def _peft_patch():
    import peft.import_utils as _peft_utils
    import peft.tuners.lora.torchao as _peft_torchao
    _orig = _peft_utils.is_torchao_available
    def _safe():
        try: return _orig()
        except ImportError: return False
    _peft_utils.is_torchao_available = _safe
    _peft_torchao.is_torchao_available = _safe


def load_pipeline(mode: str) -> None:
    """Load the pipeline for the given mode, unloading the current one first."""
    global pipe, controlnet, depth_estimator, active_mode, _satirefic_lora_loaded

    mode = mode.lower()
    if mode == active_mode:
        print(f'âœ“ {mode} already loaded.')
        return

    # â”€â”€ Unload â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _satirefic_lora_loaded = False  # reset; set True again if Satirefic reloads
    if active_mode:
        print(f'Unloading {active_mode} pipeline...')
    if pipe is not None:
        del pipe
        pipe = None
    if controlnet is not None:
        del controlnet
        controlnet = None
    if depth_estimator is not None:
        del depth_estimator
        depth_estimator = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')

    # â”€â”€ Load â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Imagine uses plain FluxPipeline (text-to-image, no ControlNet, ~18â€“19 GB VRAM).
    if mode == 'imagine':
        print('Loading FLUX.1-dev (text-to-image)...')
        pipe = FluxPipeline.from_pretrained(
            BASE_MODEL_FLUX, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode == 'reimagine':
        print('Loading FLUX.1-Kontext-dev...')
        pipe = FluxKontextPipeline.from_pretrained(
            BASE_MODEL_KONTEXT, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode in ('scene', 'portrait'):
        print('Loading ControlNet...')
        controlnet = FluxControlNetModel.from_pretrained(
            CONTROLNET_MODEL, torch_dtype=torch.bfloat16
        )
        print('Loading FLUX.1-dev...')
        pipe = FluxControlNetPipeline.from_pretrained(
            BASE_MODEL_FLUX, controlnet=controlnet, torch_dtype=torch.bfloat16
        ).to('cuda')
        if mode == 'scene':
            print('Loading depth estimator (CPU)...')
            depth_estimator = hf_pipeline(
                'depth-estimation', model=DEPTH_MODEL, device='cpu'
            )
    else:
        raise ValueError(f'Unknown mode: {mode!r}')

    # â”€â”€ LoRA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _peft_patch()
    # Gado Cartoon LoRA â”€â”€ named adapter so cartoonify() can switch between styles
    if LORA_SOURCE == 'huggingface':
        pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT,
                               adapter_name=LORA_GADO_ADAPTER_NAME)
    elif os.path.isfile(LORA_DRIVE_PATH):
        pipe.load_lora_weights(os.path.dirname(LORA_DRIVE_PATH),
                               weight_name=os.path.basename(LORA_DRIVE_PATH),
                               adapter_name=LORA_GADO_ADAPTER_NAME)
    else:
        print(f'âš ï¸  Gado Cartoon LoRA not found: {LORA_DRIVE_PATH}')

    # Satirefic LoRA â”€â”€ load only if the .safetensors file exists on Drive
    _sat_folder = os.path.dirname(LORA_SATIREFIC_PATH)
    _sat_file   = os.path.basename(LORA_SATIREFIC_PATH)
    _sat_exists = os.path.isfile(LORA_SATIREFIC_PATH)
    print(f'Satirefic LoRA path   : {LORA_SATIREFIC_PATH}')
    print(f'Satirefic path exists : {_sat_exists}')
    print(f'Satirefic folder      : {_sat_folder}')
    print(f'Satirefic file        : {_sat_file}')
    print(f'Satirefic adapter     : {LORA_SATIREFIC_ADAPTER_NAME}')
    if _sat_exists:
        pipe.load_lora_weights(_sat_folder, weight_name=_sat_file,
                               adapter_name=LORA_SATIREFIC_ADAPTER_NAME)
        _satirefic_lora_loaded = True
        print('âœ… Satirefic LoRA loaded successfully')
    else:
        _satirefic_lora_loaded = False
        print(f'âš ï¸  Satirefic LoRA not found: {LORA_SATIREFIC_PATH}')

    # Disable all adapters on load; cartoonify() activates the user-selected one.
    pipe.disable_lora()

    active_mode = mode
    print(f'âœ… {mode} pipeline ready.')


# â”€â”€ Initial load â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
load_pipeline(DEFAULT_MODE.lower())


## 8 â€” Story-to-Prompt and Wild Settings (Gemini)

Two Gemini functions:
- `build_prompt_from_story()` â€” Default mode: returns structured prompt only
- `build_wild_settings()` â€” Wild mode: returns prompt + parameter suggestions optimised for satirical impact

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEFAULT MODE — prompt only
# ─────────────────────────────────────────────────────────────────────────────

GEMINI_SYSTEM_PROMPT = """You are a prompt engineer for Satirefic — a FLUX.1 LoRA fine-tuned on grotesque anti-authoritarian newspaper caricature.
Convert the user's story into a structured image generation prompt.

OUTPUT FORMAT — one line only, no explanation, no preamble:
satirefic <medium>, <animal_cast>, <technique>, <color>, <mood>, <commentary>, <composition>

RULES:
- Always start with: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon
- ANIMAL CASTING — applies to power roles only: politicians, executives, generals, police chiefs, landlords, billionaires, diplomats, bosses.
  Choose animal by dominant trait:
    pig → greed, corruption, self-satisfaction
    vulture → exploitation, predatory capitalism
    snake → deception, manipulation
    bloated toad → pomposity, self-importance
    rooster → military vanity, strutting authority
    wolf → aggression, predation
    fat cat → landlord greed, rentier exploitation
    spider → billionaire control, web-spinning
    bulldog → police aggression, blunt enforcement
  Ordinary people — workers, protesters, civilians, the poor — always remain human.
  NEVER assign animals based on ethnicity, religion, nationality, or any protected identity.
  Animal symbolism applies exclusively to POWER ROLES, not to identity.
- Always include in technique: rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching
- Always include in color: black and white | monochrome | flat white paper | no readable text
- Pipe-separated keywords within each layer; comma between layers
- Max 5 keywords per layer
- Composition: estimate figure count, staging, contrast between animal power figures and human ordinary figures

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Prompt: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, bloated pig-politician in suit handing banknotes | human workers and families in queue | pig figure dominant, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, grotesque | scathing | bitter | dark humour | absurdist, anti-authoritarian critique | power corruption | empty promises | political condemnation | systemic satire, pig towering left | human queue stretching right | speech bubble above pig | eye level | wide shot

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Prompt: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, bloated rooster-general puffed with medals | tiny human soldiers marching on map below | rooster dominates upper frame, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, pompous | grotesque | dark | absurdist | confrontational, war metaphor | military vanity | power over lives | political condemnation | anti-authoritarian critique, rooster top-heavy upper frame | desk as throne | human soldiers lower third | top-heavy composition | high angle

Story: A massive data center consumes water, electricity, and agricultural land while executives celebrate profits.
Prompt: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, vulture-executives in suits atop smoking server stacks | dry cracked earth and drained pipes below | human farmers displaced at edges, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, dystopian | grotesque | scathing | confrontational | bitter, corporate extraction | environmental destruction | anti-authoritarian critique | systemic satire | visual irony, vulture figures celebrating upper frame | cracked land and cables lower frame | human figures pushed to edges | wide shot

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Prompt: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, two toad-diplomats shaking hands for cameras | their animal shadows fighting behind | human journalists watching at edges, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, satirical | ironic | duplicitous | darkly comic | bitter, diplomatic hypocrisy | shadow duality | anti-authoritarian critique | political irony | allegorical, toad figures foreground handshake | fighting shadows background | journalists lower corners | centred | eye level"""


def build_prompt_from_story(story: str, trigger_word: str) -> str:
    if not story.strip():
        raise ValueError('Empty story.')
    if not GOOGLE_API_KEY:
        raise ValueError('GOOGLE_API_KEY not set in Colab Secrets.')

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=GEMINI_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=500,
        ),
    )
    prompt = response.text.strip()

    # Replace trigger only when caller explicitly passes a different word
    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'satirefic':
        prompt = prompt.replace('satirefic', active_trigger, 1)

    return prompt


# ─────────────────────────────────────────────────────────────────────────────
# WILD MODE — prompt + parameter suggestions for maximum satirical impact
# ─────────────────────────────────────────────────────────────────────────────

WILD_SYSTEM_PROMPT = """You are a prompt engineer AND generation parameter optimizer for Satirefic — a FLUX.1 LoRA fine-tuned on grotesque anti-authoritarian newspaper caricature.
Your goal: craft a structured prompt AND suggest generation parameters that produce the sharpest, most satirically brutal cartoon possible.

OUTPUT FORMAT — exactly two lines, nothing else:
Line 1 (prompt): satirefic <medium>, <animal_cast>, <technique>, <color>, <mood>, <commentary>, <composition>
Line 2 (settings): {"mode": "reimagine|scene|portrait", "guidance": <2.0-6.0>, "steps": <28-40>, "cn_scale": <0.3-1.2>, "cn_end": <0.50-1.0>, "canny_low": <10-100>, "canny_high": <80-400>, "rationale": "<one sentence>"}

PROMPT RULES (same as Default):
- Start: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon
- ANIMAL CASTING — power roles only: politician → pig, executive → vulture, general → rooster, police chief → bulldog, landlord → fat cat, billionaire → spider, diplomat → toad, boss → wolf. Ordinary people remain human. Never based on protected identity.
- Always include in technique: rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching
- Always include in color: black and white | monochrome | flat white paper | no readable text
- Pipe-separated within layer, comma between layers, max 5 keywords per layer

MODE SELECTION:
- reimagine: surreal or allegorical staging, animal transformation is the main subject, identity less critical
- scene: crowd, power hierarchy, or spatial layout must be preserved
- portrait: specific face must remain recognisable through caricature — use for named individuals

PARAMETER RULES FOR SATIRICAL IMPACT:
- Confrontational / scathing / grotesque → guidance 4.5–5.5, steps 35–40
- Absurdist / ironic / allegorical → guidance 2.5–3.5, steps 28–32
- Face identity critical → portrait, canny_low 20–40, canny_high 100–160, cn_scale 0.85–1.0
- Crowd / hierarchy / multi-figure scene → scene, cn_scale 0.85–0.95, steps 34–38
- Animal transformation dominant or surreal staging → reimagine, guidance 2.5–3.5
- cn_end: 0.70 for more FLUX finishing freedom (richer ink feel), 0.80 standard, 0.85+ strict edge lock
- More figures or complex composition → more steps (36–40); single figure → 28–32

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Line 1: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, bloated pig-politician in suit handing banknotes | human workers and families in queue | pig figure dominant, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, grotesque | scathing | bitter | dark humour | absurdist, anti-authoritarian critique | power corruption | empty promises | political condemnation | systemic satire, pig towering left | human queue stretching right | speech bubble above pig | eye level | wide shot
Line 2: {"mode": "scene", "guidance": 4.5, "steps": 36, "cn_scale": 0.85, "cn_end": 0.75, "canny_low": 50, "canny_high": 200, "rationale": "Power-imbalance crowd scene — depth map locks pig-dominant layout; high guidance drives grotesque anti-authoritarian tone"}

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Line 1: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, bloated rooster-general puffed with medals | tiny human soldiers marching on map below | rooster dominates upper frame, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, pompous | grotesque | dark | absurdist | confrontational, war metaphor | military vanity | power over lives | political condemnation | anti-authoritarian critique, rooster top-heavy upper frame | desk as throne | human soldiers lower third | top-heavy composition | high angle
Line 2: {"mode": "portrait", "guidance": 5.0, "steps": 38, "cn_scale": 0.90, "cn_end": 0.70, "canny_low": 28, "canny_high": 130, "rationale": "Rooster pomposity demands tight grotesque outlines; high guidance and steps maximise anti-authoritarian caricature brutality"}

Story: A massive data center consumes water, electricity, and agricultural land while executives celebrate profits.
Line 1: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, vulture-executives in suits atop smoking server stacks | dry cracked earth and drained pipes below | human farmers displaced at edges, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, dystopian | grotesque | scathing | confrontational | bitter, corporate extraction | environmental destruction | anti-authoritarian critique | systemic satire | visual irony, vulture figures celebrating upper frame | cracked land and cables lower frame | human figures pushed to edges | wide shot
Line 2: {"mode": "scene", "guidance": 4.8, "steps": 37, "cn_scale": 0.88, "cn_end": 0.75, "canny_low": 40, "canny_high": 180, "rationale": "Multi-layer extraction scene — depth map preserves vertical hierarchy with vultures above cracked earth and displaced humans"}

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Line 1: satirefic crude black and white newspaper caricature | anti-authoritarian cartoon | grotesque illustration | political cartoon, two toad-diplomats shaking hands for cameras | their animal shadows fighting behind | human journalists watching at edges, rough black ink | dirty linework | ugly outlines | hand-drawn marks | scratchy cross-hatching, black and white | monochrome | flat white paper | no readable text, satirical | ironic | duplicitous | darkly comic | bitter, diplomatic hypocrisy | shadow duality | anti-authoritarian critique | political irony | allegorical, toad figures foreground handshake | fighting shadows background | journalists lower corners | centred | eye level
Line 2: {"mode": "reimagine", "guidance": 3.0, "steps": 32, "cn_scale": 0.70, "cn_end": 0.80, "canny_low": 50, "canny_high": 200, "rationale": "Shadow-duality allegory with toad transformation — Kontext recomposition handles dual-reality staging freely"}"""


def build_wild_settings(story: str, trigger_word: str) -> tuple:
    """Returns (prompt_str, settings_dict) for Wild mode.
    settings_dict keys: mode, guidance, steps, cn_scale, cn_end, canny_low, canny_high, rationale.
    Returns (DEFAULT_PROMPT, {}) on any failure.
    """
    if not story.strip() or not GOOGLE_API_KEY:
        return DEFAULT_PROMPT, {}

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=WILD_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=500,
        ),
    )

    lines = [l.strip() for l in response.text.strip().split('\n') if l.strip()]
    prompt = lines[0] if lines else DEFAULT_PROMPT

    # Replace trigger only when caller explicitly passes a different word
    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'satirefic':
        prompt = prompt.replace('satirefic', active_trigger, 1)

    settings = {}
    for line in lines[1:]:
        if line.startswith('{'):
            try:
                settings = json.loads(line)
            except json.JSONDecodeError:
                pass
            break

    return prompt, settings


print('✓ Default build_prompt_from_story ready')
print('✓ Wild build_wild_settings ready')


## 9 â€” Inference

In [ ]:
# ── Status HTML helpers ───────────────────────────────────────────────────────
def _status(msg: str, state: str = 'processing') -> str:
    colors = {'idle': '#3f3f46', 'processing': '#a78bfa', 'done': '#34d399', 'warn': '#fbbf24'}
    c    = colors.get(state, colors['processing'])
    anim = ' animation:dot-pulse 1.2s ease-in-out infinite;' if state == 'processing' else ''
    return (
        f'<div class="cfy-status">'
        f'<div class="cfy-dot" style="background:{c};{anim}"></div>'
        f'<span>{msg}</span>'
        f'</div>'
    )

def _progress_html(step: int, total: int) -> str:
    pct = int(100 * step / max(total, 1))
    return (
        f'<div class="cfy-status">'
        f'<div class="cfy-dot" style="background:#a78bfa;animation:dot-pulse 1.2s ease-in-out infinite;"></div>'
        f'<div class="cfy-progress-wrap">'
        f'<div class="cfy-progress-label"><span>FLUX rendering</span>'
        f'<span style="color:#a78bfa;font-weight:700">{step} / {total}</span></div>'
        f'<div class="cfy-progress-track">'
        f'<div class="cfy-progress-fill" style="width:{pct}%"></div>'
        f'</div></div></div>'
    )

_STATUS_IDLE = _status('Upload a photo, describe your story, then hit Cartoonify.', 'idle')


def extract_canny(pil_image: Image.Image, low: int, high: int) -> Image.Image:
    img_np = np.array(pil_image)
    edges  = cv2.Canny(img_np, low, high)
    edges  = edges[:, :, None]
    edges  = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


def build_final_prompt(prompt: str, lora_style: str, trigger_word: str) -> str:
    """Assemble the final prompt sent to the image model.

    Always prepends GLOBAL_PROMPT_PREFIX and the LoRA trigger word.
    Appends SATIREFIC_STYLE_BOOST when Satirefic LoRA is selected.
    Neither prefix nor boost is duplicated if already present.
    """
    base = prompt.strip()

    # Prepend LoRA trigger word (existing behavior)
    trigger = trigger_word.strip()
    if trigger and not base.startswith(trigger):
        base = f'{trigger} {base}'

    # Global prompt prefix applied to every generation
    if not base.startswith(GLOBAL_PROMPT_PREFIX):
        base = f'{GLOBAL_PROMPT_PREFIX}, {base}'

    # Satirefic hidden style boost applied only when Satirefic LoRA is selected
    if lora_style == 'Satirefic LoRA' and SATIREFIC_STYLE_BOOST not in base:
        base = f'{base}, {SATIREFIC_STYLE_BOOST}'

    return base


def cartoonify(
    story,
    image,
    mode_label,
    wild_mode,        # bool  — Checkbox value
    lora_style,       # str   — 'None' | 'Gado Cartoon' | 'Satirefic LoRA'
    lora_strength,    # float — adapter weight (0.0–1.5)
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    cn_end,
    canny_low,
    canny_high,
    seed,
    randomize_seed,
    prompt_override,
):
    """Generator. Yields (canvas, status_html, result_state, view_toggle, guidance, steps, cn_scale, cn_end, canny_low, canny_high)."""
    if image is None and mode_label.lower() != 'imagine':
        raise gr.Error('Upload a photo first.')

    mode = mode_label.lower()

    # Auto-select trigger word from the chosen LoRA style (overrides manual input)
    style_cfg = LORA_STYLES.get(lora_style, {'adapter': None, 'trigger': ''})
    if style_cfg.get('trigger'):
        trigger_word = style_cfg['trigger']

    def emit(canvas=None, status=None, result=None, show_toggle=None,
             g=None, s=None, cn_s=None, cn_e=None, clow=None, chigh=None, seed=None):
        if show_toggle is None:
            toggle_upd = gr.update()
        elif show_toggle:
            toggle_upd = gr.update(visible=True, value='Cartoon')
        else:
            toggle_upd = gr.update(visible=False)
        return (
            gr.update() if canvas is None else gr.update(value=canvas, visible=True),
            gr.update() if status is None else gr.update(value=status),
            gr.update() if result is None else result,
            toggle_upd,
            gr.update() if g     is None else gr.update(value=g),
            gr.update() if s     is None else gr.update(value=s),
            gr.update() if cn_s  is None else gr.update(value=cn_s),
            gr.update() if cn_e  is None else gr.update(value=cn_e),
            gr.update() if clow  is None else gr.update(value=clow),
            gr.update() if chigh is None else gr.update(value=chigh),
            gr.update(visible=False) if canvas is not None else gr.update(),
            gr.update() if seed  is None else gr.update(value=seed),
        )

    def flux_with_progress(pipeline_call, total_steps):
        """Run pipeline in background thread; yield HTML progress strings then the PIL Image."""
        step_q     = queue.Queue()
        result_box = [None]
        error_box  = [None]

        def _cb(_, step, _ts, kwargs):
            step_q.put(step + 1)
            return kwargs

        def _run():
            with torch.inference_mode():
                try:
                    result_box[0] = pipeline_call(callback_on_step_end=_cb)
                except Exception as exc:
                    error_box[0] = exc

        t = threading.Thread(target=_run, daemon=True)
        t.start()

        while t.is_alive():
            try:
                step = step_q.get(timeout=0.3)
                yield _progress_html(step, total_steps)
            except queue.Empty:
                pass

        t.join()
        if error_box[0]:
            raise error_box[0]
        yield result_box[0]  # final item is the PIL Image

    # Working parameter values
    g_val     = guidance_scale
    s_val     = int(num_steps)
    cn_s_val  = cn_scale
    cn_e_val  = cn_end
    clow_val  = int(canny_low)
    chigh_val = int(canny_high)

    # ── Step 1: Prompt ────────────────────────────────────────────────────────
    if prompt_override.strip():
        prompt = prompt_override.strip()
        yield emit(status=_status('✓ Using manual prompt override', 'done'))

    elif wild_mode and story.strip():
        yield emit(status=_status('⚡ Wild — Gemini building prompt and tuning for satire…'))
        try:
            prompt, wild = build_wild_settings(story, trigger_word)

            if wild:
                g_val     = float(wild.get('guidance',  g_val))
                s_val     = int(wild.get('steps',       s_val))
                cn_s_val  = float(wild.get('cn_scale',  cn_s_val))
                cn_e_val  = float(wild.get('cn_end',    cn_e_val))
                clow_val  = int(wild.get('canny_low',   clow_val))
                chigh_val = int(wild.get('canny_high',  chigh_val))
                rationale = wild.get('rationale', '')

                mode_map  = {'imagine': 'Imagine', 'reimagine': 'Reimagine', 'scene': 'Scene', 'portrait': 'Portrait'}
                suggested = wild.get('mode', '').lower()
                mode_note = ''
                if suggested and suggested != mode:
                    mode_note = f' · Wild suggests {mode_map.get(suggested, suggested)} for this story'

                yield emit(
                    status=_status(f'✓ Wild applied — {rationale}{mode_note}', 'done'),
                    g=g_val, s=s_val,
                    cn_s=cn_s_val, cn_e=cn_e_val,
                    clow=clow_val, chigh=chigh_val,
                )
            else:
                yield emit(status=_status('✓ Prompt ready (settings parse failed — using current sliders)', 'done'))

        except Exception as exc:
            yield emit(status=_status(f'⚠️  Wild failed ({exc}) — falling back to Default', 'warn'))
            try:
                prompt = build_prompt_from_story(story, trigger_word)
            except Exception:
                prompt = DEFAULT_PROMPT

    elif story.strip():
        yield emit(status=_status('Gemini building your prompt…'))
        try:
            prompt = build_prompt_from_story(story, trigger_word)
            yield emit(status=_status('✓ Prompt ready', 'done'))
        except Exception as exc:
            yield emit(status=_status(f'⚠️  Gemini failed ({exc}) — using default prompt', 'warn'))
            prompt = DEFAULT_PROMPT

    else:
        prompt = DEFAULT_PROMPT
        yield emit(status=_status('No story — using default prompt', 'idle'))

    full_prompt = build_final_prompt(prompt, lora_style, trigger_word)
    # -- Colab debug log -------------------------------------------------------
    _prompt_len   = len(full_prompt)
    _trigger_word = trigger_word.strip()
    _feed_snippet = (full_prompt[:220] + f'… [{_prompt_len} chars]') if _prompt_len > 220 else full_prompt
    print('=' * 72)
    print(f'FULL PROMPT LENGTH : {_prompt_len} chars')
    print(f'FULL PROMPT        : {full_prompt}')
    print(f'TRIGGER WORD       : {_trigger_word!r}')
    print(f'SELECTED LoRA      : {lora_style}')
    print(f'LoRA STRENGTH      : {lora_strength:.3f}')
    print('=' * 72)

    # ── Step 2: Pipeline switch ───────────────────────────────────────────────
    if mode != active_mode:
        yield emit(status=_status(f'Switching to {mode_label} pipeline (~60–90 s)…'))
        load_pipeline(mode)
        yield emit(status=_status(f'✓ {mode_label} pipeline ready', 'done'))

    # ── LoRA adapter activation ────────────────────────────────────────────────
    active_adapter = style_cfg.get('adapter')
    if lora_style == 'Satirefic LoRA':
        if not _satirefic_lora_loaded:
            raise gr.Error(
                'Satirefic LoRA is selected but was not loaded — '
                f'check that {LORA_SATIREFIC_PATH} exists and re-run the model-load cell.'
            )
        if mode != 'imagine':
            # Non-Imagine modes: activate in main thread (pipeline runs there too)
            pipe.set_adapters([LORA_SATIREFIC_ADAPTER_NAME], adapter_weights=[float(lora_strength)])
            pipe.enable_lora()
            print(f'SATIREFIC ACTIVE — weight: {float(lora_strength):.3f}')
        # Imagine: deferred to _imagine_call — same thread as pipe(), exact NB06 pattern
    elif active_adapter:
        try:
            pipe.set_adapters([active_adapter], adapter_weights=[float(lora_strength)])
            pipe.enable_lora()
        except Exception as exc:
            yield emit(status=_status(
                f'⚠️  LoRA "{lora_style}" unavailable ({exc}) — running without LoRA', 'warn'))
            pipe.disable_lora()
    else:
        # None — disable only if adapters are actually active
        try:
            if pipe.get_active_adapters():
                pipe.disable_lora()
        except Exception:
            pipe.disable_lora()

    # ── Step 3: Resize ──────────────────────────────────────────────────────── (skipped for Imagine — text-to-image needs no input image)
    if mode != 'imagine':
        src = Image.fromarray(image).convert('RGB')
        src = src.resize((1024, 1024), Image.LANCZOS)
    actual_seed = int(torch.randint(0, 2**32, (1,)).item()) if randomize_seed else int(seed)
    generator = torch.Generator(device='cuda').manual_seed(actual_seed)
    yield emit(status=_status(f'seed {actual_seed} · {_feed_snippet}', 'done'))

    # ── Step 4: Inference ────────────────────────────────────────────────────
    result = None

    if mode == 'imagine':
        # Capture values for closure — avoids any late-binding or cross-thread ambiguity
        _img_style   = lora_style
        _img_weight  = float(lora_strength)
        _img_prompt  = full_prompt
        _img_seed    = actual_seed
        _img_adapter = active_adapter

        def _imagine_call(callback_on_step_end):
            """Activate LoRA in the same thread as pipe() — exact NB06 pattern."""
            if _img_style == 'Satirefic LoRA':
                # NB06 exact sequence: set_adapters → enable_lora → pipe()
                pipe.set_adapters(
                    [LORA_SATIREFIC_ADAPTER_NAME], adapter_weights=[_img_weight]
                )
                pipe.enable_lora()
                try:
                    _active = pipe.get_active_adapters()
                except Exception:
                    _active = ['(unknown)']
                print(
                    f'SATIREFIC ACTIVE ON PIPELINE {id(pipe)}'
                    f' — weight: {_img_weight:.3f}'
                )
            elif _img_adapter:
                _active = [_img_adapter]
            else:
                _active = []
            print('=' * 64)
            print(f'STYLE REQUESTED:         {_img_style}')
            print(f'PIPELINE OBJECT ID:      {id(pipe)}')
            print(f'ACTIVE ADAPTERS:         {_active}')
            print(f'EFFECTIVE LORA STRENGTH: {_img_weight:.3f}')
            print(f'PIPELINE TYPE:           {type(pipe).__name__}')
            print(f'FINAL PROMPT:            {_img_prompt}')
            print(f'SEED:                    {_img_seed}')
            print('=' * 64)
            if (_img_style == 'Satirefic LoRA'
                    and LORA_SATIREFIC_ADAPTER_NAME not in _active):
                raise RuntimeError(
                    f'Adapter verification FAILED: '
                    f'{LORA_SATIREFIC_ADAPTER_NAME!r} not in {_active}. '
                    'Cannot generate — Satirefic is selected but not active.'
                )
            return pipe(
                prompt=_img_prompt,
                width=1024,
                height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                generator=generator,
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0]

        yield emit(status=_status('FLUX text-to-image rendering…'))
        for item in flux_with_progress(_imagine_call, s_val):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'reimagine':
        yield emit(status=_status('FLUX Kontext rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                image=src,
                prompt=full_prompt,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'scene':
        yield emit(status=_status('Reading scene depth…'))
        depth_out  = depth_estimator(src)
        depth_arr  = np.array(depth_out['depth'])
        d_min, d_max = depth_arr.min(), depth_arr.max()
        if d_max > d_min:
            depth_norm = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
        else:
            depth_norm = np.zeros_like(depth_arr, dtype=np.uint8)
        control_image = Image.fromarray(depth_norm).convert('RGB')
        yield emit(status=_status('✓ Depth map ready', 'done'))
        yield emit(status=_status('FLUX rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=control_image,
                control_mode=2,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'portrait':
        yield emit(status=_status('Extracting portrait outlines…'))
        canny_image = extract_canny(src, clow_val, chigh_val)
        yield emit(status=_status('✓ Outlines extracted', 'done'))
        yield emit(status=_status('FLUX rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=canny_image,
                control_mode=0,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                control_guidance_end=cn_e_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    if result is None:
        raise gr.Error('Generation produced no image — check the activity log above.')

    import hashlib as _hl
    _px_hash  = _hl.sha256(result.tobytes()).hexdigest()
    ts        = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix    = {'imagine': 'txt2img', 'reimagine': 'kontext', 'scene': 'depth', 'portrait': 'canny'}[mode]
    _out_path = f'{OUTPUT_DIR}/{ts}_cartoonify_{suffix}.png'
    result.save(_out_path)
    with open(_out_path, 'rb') as _fh:
        _file_hash = _hl.sha256(_fh.read()).hexdigest()
    print('--- GENERATION RESULT ---')
    print(f'pixel SHA-256 : {_px_hash}')
    print(f'file  SHA-256 : {_file_hash}')
    print(f'output file   : {_out_path}')

    gc.collect()
    torch.cuda.empty_cache()

    yield emit(
        canvas=result,
        status=_status(f'✓ Done — saved to Drive · seed {actual_seed} · {_feed_snippet}', 'done'),
        result=result,
        show_toggle=True,
        seed=actual_seed,
    )


print('✓ Inference functions ready')


In [ ]:
# â”€â”€ A/B Test â€” Satirefic LoRA weight effect on Imagine output â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import hashlib as _ab_hl
import uuid     as _ab_uuid

_AB_PROMPT_RAW = 'corrupt politician in a giant luxury tower'
_AB_SEED       = 42
_AB_STRENGTHS  = [0.3, 1.2]

# Build full prompt (mirrors build_final_prompt for Satirefic)
_ab_full_prompt = (
    f'{GLOBAL_PROMPT_PREFIX}, satirefic {_AB_PROMPT_RAW}, {SATIREFIC_STYLE_BOOST}'
)

print('=== A/B TEST: Satirefic weight effect on Imagine output ===')
print(f'Prompt : {_ab_full_prompt[:100]}...')
print(f'Seed   : {_AB_SEED}')
print(f'Runs   : strength {_AB_STRENGTHS[0]}  vs  strength {_AB_STRENGTHS[1]}')
print()

if not _satirefic_lora_loaded:
    print('SKIP: Satirefic LoRA not loaded â€” run cell-models first.')
elif active_mode != 'imagine':
    print(f'SKIP: active_mode is {active_mode!r} â€” run load_pipeline("imagine") first.')
else:
    _ab_results = {}

    for _ab_w in _AB_STRENGTHS:
        print(f'--- Strength {_ab_w} ---')
        # Exact NB06 activation: set_adapters â†’ enable_lora â†’ pipe()
        pipe.set_adapters([LORA_SATIREFIC_ADAPTER_NAME], adapter_weights=[_ab_w])
        pipe.enable_lora()
        print(f'  SATIREFIC ACTIVE ON PIPELINE {id(pipe)} â€” weight: {_ab_w}')

        _ab_gen = torch.Generator(device='cuda').manual_seed(_AB_SEED)

        with torch.inference_mode():
            _ab_out = pipe(
                prompt=_ab_full_prompt,
                width=1024, height=1024,
                num_inference_steps=28,
                guidance_scale=3.5,
                generator=_ab_gen,
                max_sequence_length=512,
            )

        _ab_img   = _ab_out.images[0]
        _ab_px    = _ab_hl.sha256(_ab_img.tobytes()).hexdigest()
        _ab_uid   = _ab_uuid.uuid4().hex[:8]
        _ab_wstr  = str(_ab_w).replace('.', 'p')
        _ab_fname = f'05_ab_strength{_ab_wstr}_seed{_AB_SEED}_{_ab_uid}.png'
        _ab_path  = f'{OUTPUT_DIR}/{_ab_fname}'
        _ab_img.save(_ab_path)
        with open(_ab_path, 'rb') as _ab_fh:
            _ab_fhash = _ab_hl.sha256(_ab_fh.read()).hexdigest()

        print(f'  pixel SHA-256 : {_ab_px[:40]}...')
        print(f'  file  SHA-256 : {_ab_fhash[:40]}...')
        print(f'  saved         : {_ab_fname}')

        _ab_results[_ab_w] = {'px': _ab_px, 'fhash': _ab_fhash}
        pipe.disable_lora()
        print()

    if len(_ab_results) == 2:
        _h03 = _ab_results[0.3]['px']
        _h12 = _ab_results[1.2]['px']
        print('=== RESULT ===')
        print(f'  strength 0.3  pixel: {_h03[:40]}...')
        print(f'  strength 1.2  pixel: {_h12[:40]}...')
        print()
        if _h03 == _h12:
            print('FAIL: LoRA strength did not affect the generated output.')
            print('  Both strengths produced the same pixel hash.')
        else:
            print('PASS: strength 0.3 and 1.2 produced different pixel hashes.')
            print('  The LoRA weight is reaching and affecting the pipeline.')


## 10 â€” Launch Interface

Re-run this cell to restart the UI without reloading models.

**Layout:** Fixed left sidebar (mode, Wild toggle, fine-tune) + right canvas area (animated placeholder â†’ uploaded photo â†’ cartoon result).

**Wild mode (âš¡ toggle):** Gemini reads the story and suggests parameters optimised for maximum satirical impact; sliders update to reflect what's being used.

**Original / Cartoon toggle:** Appears after generation completes â€” click to compare your uploaded photo against the cartoon result.


In [ ]:
MODE_DESCRIPTIONS = {
    'Imagine': (
        'FLUX generates from text only — no input photo needed. '
        'The story and prompt define the full composition. '
        'Best for new scenes with no reference image.'
    ),
    'Reimagine': (
        'FLUX recomposes the full image freely. '
        'The scene, staging, background and framing can all shift. '
        'Best for bold creative reinterpretation where you want maximum satirical transformation.'
    ),
    'Scene': (
        'A depth map locks the spatial layout and figure positions. '
        'Foreground and background relationships stay intact. '
        'Best for crowd scenes, protests, or architecture where structure must be preserved.'
    ),
    'Portrait': (
        'Canny edge detection traces face and body outlines before rendering. '
        'The person stays immediately recognisable in the cartoon. '
        'Best when satirising a specific individual — politician, CEO, public figure.'
    ),
}


def update_mode(mode):
    d             = DEFAULTS[mode.lower()]
    no_controlnet = mode in ('Imagine', 'Reimagine')  # neither uses ControlNet sliders
    is_portrait   = (mode == 'Portrait')
    return (
        MODE_DESCRIPTIONS[mode],
        gr.update(value=d['guidance']),
        gr.update(visible=not no_controlnet, value=d['cn_scale']),
        gr.update(visible=is_portrait,       value=d['cn_end']),
        gr.update(visible=is_portrait,       value=d['canny_low']),
        gr.update(visible=is_portrait,       value=d['canny_high']),
    )


def on_image_upload(image):
    if image is None:
        return (gr.update(visible=True), gr.update(value=None, visible=False),
                None, gr.update(visible=False), None)
    pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    return (gr.update(visible=False), gr.update(value=pil, visible=True),
            pil, gr.update(visible=False), None)


def toggle_view(choice, original, result):
    img = original if choice == 'Original' else result
    return gr.update(value=img)


CANVAS_PLACEHOLDER = '''
<div class="cfy-empty">
    <div class="cfy-empty-glyph">&#9398;</div>
    <div class="cfy-empty-tagline">// VISUAL INSURGENCY STANDBY //</div>
    <div class="cfy-ticker-outer">
        <div class="cfy-ticker-inner">
            <div class="cfy-quote">
                <p>&ldquo;Satire is the weapon of the powerless against the powerful.&rdquo;</p>
                <span>&mdash; Molly Ivins</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Comedy is simply a funny way of being serious.&rdquo;</p>
                <span>&mdash; Peter Ustinov</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;A caricature puts the face of a joke on the body of a truth.&rdquo;</p>
                <span>&mdash; Joseph Conrad</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;The purpose of satire is to strip away the veneer of comforting illusion.&rdquo;</p>
                <span>&mdash; Michael Foot</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Through laughter we hold up a mirror to the absurd.&rdquo;</p>
                <span>&mdash; Cartoonify</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Laughter is the shortest distance between two people.&rdquo;</p>
                <span>&mdash; Victor Borge</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Satire is the weapon of the powerless against the powerful.&rdquo;</p>
                <span>&mdash; Molly Ivins</span>
            </div>
        </div>
    </div>
    <p class="cfy-empty-hint">&gt;_ UPLOAD SOURCE MATERIAL TO INITIALIZE</p>
</div>
'''

CSS = '''
/* ── Tokens ────────────────────────────────────────────────── */
:root {
    --bg:           #0a0a0b;
    --surface:      #111114;
    --surface-2:    #1a1a1f;
    --border:       #2a2a36;
    --border-hi:    #3e3e52;
    --acid:         #39ff14;
    --acid-dim:     rgba(57,255,20,0.10);
    --acid-glow:    rgba(57,255,20,0.28);
    --magenta:      #ff2d9b;
    --magenta-dim:  rgba(255,45,155,0.12);
    --cyan:         #00ffe7;
    --cyan-dim:     rgba(0,255,231,0.10);
    --orange:       #ff6b00;
    --orange-dim:   rgba(255,107,0,0.12);
    --text:         #e8e8e0;
    --muted:        #5a5a6e;
    --radius:       5px;
    --mono:         ui-monospace, "Courier New", Courier, monospace;
}

/* ── Page reset ─────────────────────────────────────────────── */
.gradio-container {
    font-family: var(--mono) !important;
    background: var(--bg) !important;
    max-width: 100% !important;
    margin: 0 !important;
    padding: 0 !important;
    color: var(--text) !important;
}
.gradio-container * { box-sizing: border-box; }
.dark, body, .main, .wrap, .contain, .block { background: transparent !important; }

/* ── Scanline overlay ───────────────────────────────────────── */
.gradio-container::before {
    content: "";
    position: fixed;
    inset: 0;
    background: repeating-linear-gradient(
        0deg,
        transparent,
        transparent 2px,
        rgba(0,0,0,0.07) 2px,
        rgba(0,0,0,0.07) 4px
    );
    pointer-events: none;
    z-index: 9999;
}

/* ── Outer row ──────────────────────────────────────────────── */
#cfy-app { gap: 0 !important; }
#cfy-app > .wrap { gap: 0 !important; }

/* ── SIDEBAR ─────────────────────────────────────────────────── */
#cfy-sidebar {
    min-width: 280px !important;
    max-width: 280px !important;
    background: var(--surface) !important;
    border-right: 1px solid var(--acid) !important;
    padding: 1rem !important;
    overflow-y: auto !important;
    align-self: stretch !important;
}

/* ── Header ──────────────────────────────────────────────────── */
.cfy-header {
    padding-bottom: 0.8rem;
    border-bottom: 1px solid var(--border-hi);
    margin-bottom: 0.7rem;
}
.cfy-logo {
    display: block;
    font-family: var(--mono);
    font-size: 1.3rem;
    font-weight: 900;
    letter-spacing: 0.07em;
    text-transform: uppercase;
    color: var(--acid);
    text-shadow: 0 0 14px var(--acid-glow), 2px 0 0 var(--magenta);
    animation: cfy-glitch 8s ease-in-out infinite;
    user-select: none;
}
@keyframes cfy-glitch {
    0%, 88%, 100% {
        text-shadow: 0 0 14px var(--acid-glow), 2px 0 0 var(--magenta);
        transform: none;
    }
    90% {
        text-shadow: -2px 0 0 var(--cyan), 2px 0 0 var(--magenta), 0 0 20px var(--acid-glow);
        transform: skewX(-0.5deg);
    }
    92% {
        text-shadow: 2px 0 0 var(--cyan), -2px 0 0 var(--magenta);
        transform: skewX(0.3deg);
    }
    94% {
        text-shadow: 0 0 14px var(--acid-glow), 2px 0 0 var(--magenta);
        transform: none;
    }
}
.cfy-subtitle {
    display: block;
    font-family: var(--mono);
    font-size: 0.56rem;
    letter-spacing: 0.09em;
    text-transform: uppercase;
    color: var(--muted);
    margin-top: 0.25rem;
    line-height: 1.4;
}

/* ── Animated system-status ticker ─────────────────────────── */
.cfy-sys-status {
    display: flex;
    align-items: center;
    gap: 0.45rem;
    padding: 0.3rem 0.5rem;
    margin-top: 0.55rem;
    background: var(--surface-2);
    border: 1px solid var(--border);
    border-left: 2px solid var(--acid);
    border-radius: var(--radius);
    overflow: hidden;
    height: 1.6rem;
}
.cfy-sys-dot {
    width: 5px;
    height: 5px;
    border-radius: 50%;
    background: var(--acid);
    flex-shrink: 0;
    animation: sys-blink 2.4s ease-in-out infinite;
}
@keyframes sys-blink {
    0%, 100% { opacity: 1; box-shadow: 0 0 5px var(--acid); }
    50%       { opacity: 0.35; box-shadow: none; }
}
.cfy-sys-ticker-wrap {
    flex: 1;
    overflow: hidden;
    height: 0.95rem;
}
.cfy-sys-ticker {
    animation: sys-scroll 9s linear infinite;
}
.cfy-sys-line {
    height: 0.95rem;
    font-family: var(--mono);
    font-size: 0.58rem;
    font-weight: 800;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: var(--acid);
    display: flex;
    align-items: center;
}
@keyframes sys-scroll {
    0%,   18% { transform: translateY(0); }
    25%,  43% { transform: translateY(-0.95rem); }
    50%,  68% { transform: translateY(-1.9rem); }
    75%,  93% { transform: translateY(-2.85rem); }
    100%       { transform: translateY(-3.8rem); }
}

/* ── Section labels ──────────────────────────────────────────── */
.cfy-section-label {
    display: block;
    font-family: var(--mono);
    font-size: 0.58rem;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.13em;
    color: var(--cyan);
    margin: 0.9rem 0 0.35rem;
}
.cfy-section-label::before {
    content: ">_ ";
    color: var(--acid);
}

/* ── Mode selector — sticker-style tabs ─────────────────────── */
#cfy-mode-nav { background: transparent !important; border: none !important; padding: 0 !important; }
#cfy-mode-nav .wrap { display: grid !important; grid-template-columns: 1fr 1fr !important; gap: 4px !important; }
#cfy-mode-nav label {
    padding: 0.55rem 0.35rem !important;
    border-radius: 3px !important;
    border: 1px solid var(--border-hi) !important;
    background: var(--surface-2) !important;
    color: var(--muted) !important;
    font-family: var(--mono) !important;
    font-weight: 800 !important;
    font-size: 0.68rem !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
    text-align: center !important;
    cursor: pointer !important;
    transition: border-color 0.12s, color 0.12s, background 0.12s, box-shadow 0.12s !important;
}
#cfy-mode-nav label:has(input:checked) {
    border-color: var(--acid) !important;
    border-width: 2px !important;
    background: var(--acid-dim) !important;
    color: var(--acid) !important;
    box-shadow: 0 0 10px var(--acid-glow) !important;
    text-shadow: 0 0 8px var(--acid-glow) !important;
}
#cfy-mode-nav label:hover:not(:has(input:checked)) {
    border-color: var(--border-hi) !important;
    color: var(--text) !important;
    background: rgba(255,255,255,0.03) !important;
}
#cfy-mode-nav input[type="radio"] { display: none !important; }

/* Mode description */
#cfy-mode-desc { margin-top: 0.3rem !important; margin-bottom: 0.4rem !important; }
#cfy-mode-desc p {
    color: var(--muted) !important;
    font-family: var(--mono) !important;
    font-size: 0.68rem !important;
    line-height: 1.55 !important;
    margin: 0 !important;
    padding: 0.25rem 0.5rem !important;
    border-left: 2px solid var(--border-hi);
}

/* ── Wild toggle ─────────────────────────────────────────────── */
#cfy-wild {
    padding: 0.45rem 0.6rem !important;
    border-radius: var(--radius) !important;
    background: var(--surface-2) !important;
    border: 1px solid var(--border) !important;
    border-left: 2px solid var(--orange) !important;
    transition: background 0.15s, border-color 0.15s !important;
}
#cfy-wild:has(input:checked) {
    background: var(--orange-dim) !important;
    border-color: var(--orange) !important;
    box-shadow: 0 0 10px rgba(255,107,0,0.15) !important;
}
#cfy-wild label {
    color: var(--text) !important;
    font-family: var(--mono) !important;
    font-weight: 800 !important;
    font-size: 0.7rem !important;
    letter-spacing: 0.06em !important;
    text-transform: uppercase !important;
    cursor: pointer !important;
    gap: 0.5rem !important;
}
#cfy-wild input[type="checkbox"] {
    appearance: none !important;
    -webkit-appearance: none !important;
    width: 2rem !important;
    height: 1rem !important;
    background: var(--border) !important;
    border-radius: 999px !important;
    position: relative !important;
    cursor: pointer !important;
    flex-shrink: 0 !important;
    transition: background 0.2s !important;
}
#cfy-wild input[type="checkbox"]:checked { background: var(--orange) !important; }
#cfy-wild input[type="checkbox"]::after {
    content: "" !important;
    position: absolute !important;
    top: 2px !important; left: 2px !important;
    width: 12px !important; height: 12px !important;
    background: white !important;
    border-radius: 50% !important;
    transition: transform 0.2s !important;
}
#cfy-wild input[type="checkbox"]:checked::after { transform: translateX(16px) !important; }

/* ── LoRA selector ───────────────────────────────────────────── */
#cfy-lora-nav { background: transparent !important; border: none !important; padding: 0 !important; }
#cfy-lora-nav .wrap { flex-direction: column !important; gap: 2px !important; }
#cfy-lora-nav label {
    padding: 0.38rem 0.5rem !important;
    border-radius: var(--radius) !important;
    border: 1px solid var(--border) !important;
    background: var(--surface-2) !important;
    color: var(--muted) !important;
    font-family: var(--mono) !important;
    font-weight: 700 !important;
    font-size: 0.68rem !important;
    letter-spacing: 0.05em !important;
    cursor: pointer !important;
    transition: border-color 0.12s, color 0.12s, background 0.12s !important;
}
#cfy-lora-nav label:has(input:checked) {
    border-color: var(--magenta) !important;
    background: var(--magenta-dim) !important;
    color: var(--magenta) !important;
    text-shadow: 0 0 6px rgba(255,45,155,0.25) !important;
}
#cfy-lora-nav label:hover:not(:has(input:checked)) {
    border-color: var(--border-hi) !important;
    color: var(--text) !important;
}
#cfy-lora-nav input[type="radio"] { display: none !important; }

/* ── Accordion ───────────────────────────────────────────────── */
.gradio-container details {
    background: transparent !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    margin-top: 0.5rem !important;
}
.gradio-container details summary {
    color: var(--muted) !important;
    font-family: var(--mono) !important;
    font-size: 0.68rem !important;
    font-weight: 800 !important;
    letter-spacing: 0.1em !important;
    text-transform: uppercase !important;
    padding: 0.42rem 0.6rem !important;
    cursor: pointer !important;
}
.gradio-container details summary:hover { color: var(--cyan) !important; }
.gradio-container details[open] summary {
    color: var(--cyan) !important;
    border-bottom: 1px solid var(--border) !important;
}
.gradio-container details .block { padding: 0.6rem !important; }

/* ── Sliders ─────────────────────────────────────────────────── */
.gradio-container input[type="range"] { accent-color: var(--acid) !important; }
.gradio-container label span {
    color: var(--muted) !important;
    font-family: var(--mono) !important;
    font-size: 0.67rem !important;
    text-transform: uppercase !important;
    letter-spacing: 0.07em !important;
}

/* ── Canvas area ─────────────────────────────────────────────── */
#cfy-canvas-area {
    padding: 1rem !important;
    min-width: 0 !important;
}
#cfy-canvas-area > .wrap { display: flex; flex-direction: column; gap: 0.6rem; }

/* ── Empty canvas placeholder ────────────────────────────────── */
#cfy-canvas-empty {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    min-height: 490px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    padding: 0 !important;
    position: relative !important;
    overflow: hidden !important;
}
#cfy-canvas-empty::after {
    content: "";
    position: absolute;
    inset: 0;
    background: repeating-linear-gradient(
        0deg,
        transparent,
        transparent 3px,
        rgba(57,255,20,0.013) 3px,
        rgba(57,255,20,0.013) 4px
    );
    pointer-events: none;
}
.cfy-empty {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 1.2rem;
    padding: 2rem;
    width: 100%;
    height: 100%;
    position: relative;
    z-index: 1;
}
.cfy-empty-glyph {
    font-size: 2rem;
    opacity: 0.1;
    color: var(--acid);
    user-select: none;
}
.cfy-empty-tagline {
    font-family: var(--mono);
    font-size: 0.62rem;
    font-weight: 800;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: var(--acid);
    opacity: 0.45;
}

/* Quotes ticker */
.cfy-ticker-outer {
    width: 100%;
    max-width: 560px;
    height: 5.5rem;
    overflow: hidden;
    position: relative;
    mask-image: linear-gradient(to bottom, transparent, black 22%, black 78%, transparent);
    -webkit-mask-image: linear-gradient(to bottom, transparent, black 22%, black 78%, transparent);
}
.cfy-ticker-inner { animation: ticker-scroll 28s linear infinite; }
.cfy-quote {
    height: 5.5rem;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 0.35rem;
    padding: 0 1rem;
    text-align: center;
}
.cfy-quote p {
    font-family: var(--mono);
    font-size: 0.83rem;
    color: var(--text);
    font-weight: 500;
    line-height: 1.55;
    margin: 0 !important;
    opacity: 0.62;
}
.cfy-quote span {
    font-family: var(--mono);
    font-size: 0.67rem;
    color: var(--muted);
    letter-spacing: 0.05em;
}
@keyframes ticker-scroll {
    0%   { transform: translateY(0); }
    100% { transform: translateY(calc(-5.5rem * 6)); }
}
.cfy-empty-hint {
    font-family: var(--mono);
    font-size: 0.62rem;
    color: var(--muted);
    letter-spacing: 0.09em;
    text-transform: uppercase;
    margin: 0;
}

/* ── Canvas image ────────────────────────────────────────────── */
#cfy-canvas {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
}
#cfy-canvas img { border-radius: 3px !important; object-fit: contain !important; }

/* ── View toggle ─────────────────────────────────────────────── */
#cfy-toggle {
    width: 100% !important;
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 3px !important;
}
#cfy-toggle .wrap { display: flex !important; gap: 3px !important; width: 100% !important; }
#cfy-toggle label {
    flex: 1 !important;
    text-align: center !important;
    padding: 0.4rem 0 !important;
    border-radius: 3px !important;
    font-family: var(--mono) !important;
    font-size: 0.72rem !important;
    font-weight: 800 !important;
    letter-spacing: 0.09em !important;
    text-transform: uppercase !important;
    color: var(--muted) !important;
    cursor: pointer !important;
    transition: background 0.12s, color 0.12s !important;
    border: none !important;
    background: transparent !important;
}
#cfy-toggle label:has(input:checked) {
    background: var(--acid) !important;
    color: #0a0a0b !important;
}
#cfy-toggle input[type="radio"] { display: none !important; }

/* ── Status / activity box ───────────────────────────────────── */
#cfy-status-box {
    background: var(--surface-2) !important;
    border: 1px solid var(--border) !important;
    border-left: 2px solid var(--cyan) !important;
    border-radius: var(--radius) !important;
    padding: 0.4rem 0.7rem !important;
    min-height: 2.4rem !important;
}
.cfy-status {
    display: flex;
    align-items: center;
    gap: 0.5rem;
    min-height: 1.35rem;
    padding: 0;
    font-family: var(--mono);
    font-size: 0.7rem;
    color: var(--muted);
}
.cfy-dot { width: 6px; height: 6px; border-radius: 50%; flex-shrink: 0; }
.cfy-progress-wrap { flex: 1; display: flex; flex-direction: column; gap: 3px; }
.cfy-progress-label {
    display: flex;
    justify-content: space-between;
    font-family: var(--mono);
    font-size: 0.7rem;
    color: var(--muted);
}
.cfy-progress-track { height: 2px; background: var(--border); border-radius: 2px; overflow: hidden; }
.cfy-progress-fill { height: 100%; background: var(--acid); border-radius: 2px; transition: width 0.08s linear; }
@keyframes dot-pulse {
    0%, 100% { opacity: 1; transform: scale(1); }
    50%       { opacity: 0.3; transform: scale(0.7); }
}

/* ── Input bar ───────────────────────────────────────────────── */
#cfy-input-row {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 0.65rem !important;
    gap: 0.65rem !important;
}
#cfy-input-row > .wrap { align-items: stretch !important; }

/* ── Text inputs ─────────────────────────────────────────────── */
.gradio-container textarea,
.gradio-container input[type="text"],
.gradio-container input[type="number"] {
    background: var(--surface-2) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    color: var(--text) !important;
    font-family: var(--mono) !important;
    font-size: 0.8rem !important;
}
.gradio-container textarea::placeholder,
.gradio-container input::placeholder {
    color: var(--muted) !important;
    font-family: var(--mono) !important;
}
.gradio-container textarea:focus,
.gradio-container input:focus {
    border-color: var(--cyan) !important;
    outline: none !important;
    box-shadow: 0 0 0 2px var(--cyan-dim) !important;
}

/* ── Photo upload ────────────────────────────────────────────── */
.cfy-upload {
    border: 1px dashed var(--border-hi) !important;
    border-radius: var(--radius) !important;
    background: var(--surface-2) !important;
    transition: border-color 0.15s !important;
    overflow: hidden !important;
}
.cfy-upload:hover { border-color: var(--acid) !important; }
.cfy-upload .wrap { overflow: hidden !important; height: 100% !important; }
.cfy-upload p, .cfy-upload .upload-text, .cfy-upload span.or { display: none !important; }
.cfy-upload button.upload { display: none !important; }
.cfy-upload svg { width: 18px !important; height: 18px !important; opacity: 0.3 !important; }

/* ── Generate button ─────────────────────────────────────────── */
#cfy-btn button {
    background: #0a0a0b !important;
    color: var(--acid) !important;
    border: 1px solid var(--acid) !important;
    border-radius: var(--radius) !important;
    font-family: var(--mono) !important;
    font-size: 0.88rem !important;
    font-weight: 900 !important;
    letter-spacing: 0.14em !important;
    text-transform: uppercase !important;
    height: 46px !important;
    width: 100% !important;
    box-shadow: 0 0 14px var(--acid-glow), inset 0 0 10px rgba(57,255,20,0.04) !important;
    transition: box-shadow 0.15s, transform 0.1s !important;
    cursor: pointer !important;
}
#cfy-btn button:hover {
    box-shadow: 0 0 28px var(--acid-glow), 0 0 52px rgba(57,255,20,0.10), inset 0 0 14px rgba(57,255,20,0.06) !important;
    animation: btn-jitter 0.14s ease-in-out 1 !important;
}
@keyframes btn-jitter {
    0%   { transform: translateX(0) translateY(0); }
    30%  { transform: translateX(-1px) translateY(0.5px); }
    70%  { transform: translateX(1px) translateY(-0.5px); }
    100% { transform: translateX(0) translateY(0); }
}
#cfy-btn button:active { transform: scale(0.98) !important; box-shadow: 0 0 8px var(--acid-glow) !important; }
#cfy-btn button:disabled { opacity: 0.38 !important; cursor: wait !important; animation: none !important; }

footer { display: none !important; }

/* ── Moving scanlines ───────────────────────────────────────── */
@keyframes scanlines-drift {
    0%   { background-position: 0 0; }
    100% { background-position: 0 4px; }
}
.gradio-container::before {
    animation: scanlines-drift 0.18s linear infinite !important;
}

/* ── Grain / noise drift ────────────────────────────────────── */
@keyframes grain-drift {
    0%   { background-position: 0 0, 0 0; }
    100% { background-position: 60px 60px, -60px 60px; }
}
body::after {
    content: "";
    position: fixed;
    inset: 0;
    background-image:
        repeating-linear-gradient(
            47deg,
            transparent 0px, transparent 1px,
            rgba(255,255,255,0.007) 1px, rgba(255,255,255,0.007) 2px
        ),
        repeating-linear-gradient(
            -43deg,
            transparent 0px, transparent 1px,
            rgba(255,255,255,0.005) 1px, rgba(255,255,255,0.005) 2px
        );
    pointer-events: none;
    z-index: 9996;
    animation: grain-drift 14s linear infinite;
}

/* ── Sidebar acid glow / spray overspray ────────────────────── */
#cfy-sidebar {
    box-shadow:
        4px 0 14px rgba(57,255,20,0.07),
        12px 0 28px rgba(57,255,20,0.03) !important;
}

/* ── Paint drip on logo ─────────────────────────────────────── */
.cfy-logo {
    position: relative !important;
}
.cfy-logo::after {
    content: "";
    position: absolute;
    bottom: -3px;
    left: 44%;
    width: 1.5px;
    height: 0;
    background: linear-gradient(to bottom, var(--acid), rgba(57,255,20,0.15));
    border-radius: 0 0 2px 2px;
    animation: logo-drip 10s ease-in infinite 4s;
}
@keyframes logo-drip {
    0%, 60%, 100% { height: 0; opacity: 0; }
    65%            { height: 0; opacity: 1; }
    87%            { height: 13px; opacity: 1; }
    96%, 100%     { height: 13px; opacity: 0; }
}

/* ── Mode tab hover tilt ────────────────────────────────────── */
#cfy-mode-nav label {
    transition: transform 0.11s ease, border-color 0.12s, color 0.12s,
                background 0.12s, box-shadow 0.12s !important;
}
#cfy-mode-nav label:hover:not(:has(input:checked)) {
    transform: rotate(-0.7deg) translateY(-1px) scale(1.02) !important;
}
#cfy-mode-nav label:has(input:checked) {
    transform: none !important;
}

/* ── LoRA selector sticker edges ────────────────────────────── */
#cfy-lora-nav label:has(input:checked) {
    box-shadow:
        0 0 0 1px var(--magenta),
        1px 1px 0 1px rgba(255,45,155,0.30),
        -1px -1px 0 1px rgba(255,45,155,0.10) !important;
}

/* ── Button idle pulse ──────────────────────────────────────── */
@keyframes btn-idle-pulse {
    0%, 100% {
        box-shadow: 0 0 14px var(--acid-glow), inset 0 0 10px rgba(57,255,20,0.04);
    }
    50% {
        box-shadow: 0 0 28px var(--acid-glow), 0 0 52px rgba(57,255,20,0.11),
                    inset 0 0 14px rgba(57,255,20,0.06);
    }
}
#cfy-btn button:not(:disabled) {
    animation: btn-idle-pulse 2.8s ease-in-out infinite !important;
}
#cfy-btn button:not(:disabled):hover {
    animation: btn-jitter 0.14s ease-in-out 1 !important;
}

/* ── Armed animation while generating (disabled state) ─────── */
@keyframes btn-armed {
    0%, 100% {
        border-color: var(--magenta) !important;
        color: var(--magenta) !important;
        box-shadow: 0 0 24px rgba(255,45,155,0.40),
                    inset 0 0 10px rgba(255,45,155,0.06) !important;
    }
    50% {
        border-color: var(--orange) !important;
        color: var(--orange) !important;
        box-shadow: 0 0 28px rgba(255,107,0,0.35),
                    inset 0 0 10px rgba(255,107,0,0.06) !important;
    }
}
#cfy-btn button:disabled {
    opacity: 1 !important;
    cursor: wait !important;
    animation: btn-armed 0.9s ease-in-out infinite !important;
}

/* ── Canvas atmosphere ring / shockwave ─────────────────────── */
@keyframes canvas-atmosphere {
    0%, 76%, 100% { box-shadow: 0 0 0 1px var(--border); }
    82%            { box-shadow: 0 0 0 7px rgba(0,255,231,0.16), 0 0 0 1px var(--border); }
    90%            { box-shadow: 0 0 0 2px rgba(0,255,231,0.05), 0 0 0 1px var(--border); }
}
#cfy-canvas {
    animation: canvas-atmosphere 5s ease-in-out infinite !important;
}

/* ── Prefers-reduced-motion ─────────────────────────────────── */
@media (prefers-reduced-motion: reduce) {
    *, *::before, *::after {
        animation-duration: 0.01ms !important;
        animation-iteration-count: 1 !important;
        transition-duration: 0.01ms !important;
    }
}
'''


with gr.Blocks(
    css=CSS,
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.green,
        neutral_hue=gr.themes.colors.zinc,
    ),
    title='CARTOONIFY.exe',
) as demo:

    original_state = gr.State(None)
    result_state   = gr.State(None)

    with gr.Row(elem_id='cfy-app', equal_height=False):

        # ── LEFT SIDEBAR ──────────────────────────────────────────────────────
        with gr.Column(scale=2, min_width=268, elem_id='cfy-sidebar'):

            gr.HTML('''
<div class="cfy-header">
    <span class="cfy-logo">CARTOONIFY.exe</span>
    <span class="cfy-subtitle">SATIRICAL IMAGE LAB // ARCHITECTURE, POWER, ABSURDITY</span>
    <div class="cfy-sys-status" aria-label="System status: online">
        <div class="cfy-sys-dot"></div>
        <div class="cfy-sys-ticker-wrap" aria-hidden="true">
            <div class="cfy-sys-ticker">
                <div class="cfy-sys-line">MODEL ONLINE</div>
                <div class="cfy-sys-line">SATIRE ENGINE ARMED</div>
                <div class="cfy-sys-line">PUBLIC REALITY UNSTABLE</div>
                <div class="cfy-sys-line">ARCHITECTURE UNDER REVIEW</div>
                <div class="cfy-sys-line">MODEL ONLINE</div>
            </div>
        </div>
    </div>
</div>
''')

            gr.HTML('<span class="cfy-section-label">INPUT SIGNAL</span>')
            mode_selector = gr.Radio(
                choices=['Imagine', 'Reimagine', 'Scene', 'Portrait'],
                value=DEFAULT_MODE, label='', show_label=False,
                elem_id='cfy-mode-nav',
            )
            mode_desc = gr.Markdown(
                value=MODE_DESCRIPTIONS[DEFAULT_MODE],
                elem_id='cfy-mode-desc',
            )

            gr.HTML('<span class="cfy-section-label">OPTIONS</span>')
            wild_toggle = gr.Checkbox(
                value=False,
                label='⚡ WILD — GEMINI TUNES FOR MAX SATIRE',
                elem_id='cfy-wild',
            )

            gr.HTML('<span class="cfy-section-label">STYLE MUTATOR</span>')
            # LoRA options: None = no adapter, Gado Cartoon = existing team LoRA,
            # Satirefic LoRA = personal trained LoRA at LORA_SATIREFIC_PATH
            lora_selector = gr.Radio(
                choices=['None', 'Gado Cartoon', 'Satirefic LoRA'],
                value='Gado Cartoon', label='', show_label=False,
                elem_id='cfy-lora-nav',
            )
            lora_strength_slider = gr.Slider(
                minimum=0.0, maximum=1.5, value=0.8, step=0.05,
                label='LoRA Strength',
            )

            gr.HTML('<span class="cfy-section-label">SEED</span>')
            seed_input = gr.Number(value=DEFAULT_SEED, label='Seed', precision=0)
            randomize_seed_check = gr.Checkbox(
                value=True, label='Randomize seed',
                info='ON = new variation each run; OFF = repeatable seed',
            )

            with gr.Accordion('CONTROL PANEL', open=False):
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=10.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['guidance'],
                    step=0.5, label='Guidance Scale',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                )
                cn_scale_slider = gr.Slider(
                    minimum=0.1, maximum=1.5,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_scale'],
                    step=0.05, label='ControlNet Scale',
                    visible=DEFAULT_MODE != 'Reimagine',
                )
                cn_end_slider = gr.Slider(
                    minimum=0.3, maximum=1.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_end'],
                    step=0.05, label='ControlNet Guidance End',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_low_slider = gr.Slider(
                    minimum=0, maximum=200,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_low'],
                    step=10, label='Canny Low Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_high_slider = gr.Slider(
                    minimum=50, maximum=500,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_high'],
                    step=10, label='Canny High Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                trigger_input = gr.Textbox(value=DEFAULT_TRIGGER, label='LoRA Trigger Word', lines=1)
                gr.HTML('<div class="cfy-section-label" style="margin-top:.8rem;color:var(--orange)">FULL MANUAL PROMPT // overrides Gemini</div>')
                prompt_input = gr.Textbox(
                    label='',
                    show_label=False,
                    placeholder=(
                        'Paste or type a full editorial-cartoon prompt here.\n'
                        'When set, this overrides Gemini completely -- '
                        'your exact text reaches FLUX unchanged.\n'
                        'Leave empty to let Gemini build the prompt from your story.\n'
                        'Example: satirefic crude black and white newspaper caricature | '
                        'anti-authoritarian cartoon, bloated pig-politician handing banknotes | '
                        'human workers in queue, rough black ink | dirty linework | grotesque | scathing'
                    ),
                    lines=8,
                    max_lines=16,
                    value='',
                )

        # ── CANVAS + INPUT ────────────────────────────────────────────────────
        with gr.Column(scale=7, elem_id='cfy-canvas-area'):

            # Animated quote ticker — hidden once an image is loaded
            canvas_placeholder = gr.HTML(
                value=CANVAS_PLACEHOLDER,
                elem_id='cfy-canvas-empty',
                visible=True,
            )

            # Main canvas — shows uploaded photo, then the cartoon result
            canvas_display = gr.Image(
                label='', type='pil', interactive=False,
                elem_id='cfy-canvas', height=490,
                visible=False,
            )

            # Original / Cartoon toggle — revealed after generation completes
            view_toggle = gr.Radio(
                choices=['Original', 'Cartoon'],
                value='Cartoon', label='', show_label=False,
                visible=False, elem_id='cfy-toggle',
            )

            gr.HTML('<span class="cfy-section-label" style="margin:0 0 0.2rem">OUTPUT FEED</span>')
            status_output = gr.HTML(value=_STATUS_IDLE, elem_id='cfy-status-box')

            # Bottom bar: story text + compact photo upload thumb
            with gr.Row(elem_id='cfy-input-row', equal_height=True):
                with gr.Column(scale=5):
                    story_input = gr.Textbox(
                        label='', lines=3, max_lines=6,
                        placeholder=(
                            'Describe your story or what to change…\n'
                            'e.g. A politician handing out empty promises while people queue for food'
                        ),
                    )
                with gr.Column(scale=2, min_width=130):
                    img_input = gr.Image(
                        label='Photo', type='numpy', height=104,
                        elem_classes=['cfy-upload'],
                        sources=['upload', 'clipboard'],
                    )

            generate_btn = gr.Button('GENERATE SATIRE', variant='primary', elem_id='cfy-btn')

    # ── Event wiring ──────────────────────────────────────────────────────────

    def on_lora_change(style):
        cfg = LORA_STYLES.get(style, {})
        return gr.update(value=cfg.get('trigger', ''))

    lora_selector.change(
        fn=on_lora_change,
        inputs=[lora_selector],
        outputs=[trigger_input],
    )

    mode_selector.change(
        fn=update_mode, inputs=[mode_selector],
        outputs=[mode_desc, guidance_slider, cn_scale_slider,
                 cn_end_slider, canny_low_slider, canny_high_slider],
    )

    img_input.change(
        fn=on_image_upload, inputs=[img_input],
        outputs=[canvas_placeholder, canvas_display, original_state, view_toggle, result_state],
    )

    generate_btn.click(
        fn=cartoonify,
        inputs=[
            story_input, img_input, mode_selector, wild_toggle,
            lora_selector, lora_strength_slider,
            trigger_input, guidance_slider, steps_slider,
            cn_scale_slider, cn_end_slider, canny_low_slider, canny_high_slider,
            seed_input, randomize_seed_check, prompt_input,
        ],
        outputs=[
            canvas_display, status_output, result_state,
            view_toggle,
            guidance_slider, steps_slider,
            cn_scale_slider, cn_end_slider,
            canny_low_slider, canny_high_slider,
            canvas_placeholder,
            seed_input,
        ],
        api_name='cartoonify',
    )

    view_toggle.change(
        fn=toggle_view,
        inputs=[view_toggle, original_state, result_state],
        outputs=[canvas_display],
    )


demo.launch(share=True, show_error=True, quiet=False)
